# Run Analysis — Subtask 1 Vision

Loads a single vision run from `results/subtask1/vision_runs/<RUN_ID>/` and surfaces:
- Training curves (loss, Acc±1, MAE per epoch)
- Per-class recall trajectory and best-epoch bar chart
- Confusion matrix at best epoch
- val_probs summary (if available)
- Side-by-side comparison table of all locally available runs

## Setup

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "scripts").exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Run from inside the ai4agri repo")

RESULTS_DIR  = REPO_ROOT / "results" / "subtask1"
RUNS_DIR     = RESULTS_DIR / "vision_runs"
VAL_PREDS_DIR = RESULTS_DIR / "val_preds"

CLASS_NAMES  = ["Very Low", "Low", "Medium", "High", "Very High"]
CLASS_COLORS = ["#d73027", "#fc8d59", "#fee08b", "#91cf60", "#1a9850"]

plt.rcParams.update({"figure.figsize": (14, 5), "axes.titlesize": 12,
                     "axes.labelsize": 11, "font.size": 10})

# ── Set this to the run you want to inspect ───────────────────────────────────
RUN_ID = "existing_unet_pm1bce_softnsum_summary_rand_e10_p1024_v256_m5"

RUN_DIR      = RUNS_DIR / RUN_ID
VAL_PROBS    = VAL_PREDS_DIR / f"{RUN_ID}_val_probs.npz"

print("Repo root :", REPO_ROOT)
print("Run dir   :", RUN_DIR)
print("Run dir exists:", RUN_DIR.exists())
print("val_probs  exists:", VAL_PROBS.exists())


## Load Artifacts

In [ ]:
def load_json(path):
    return json.loads(Path(path).read_text()) if Path(path).exists() else None

config  = load_json(RUN_DIR / "config.json")
metrics = load_json(RUN_DIR / "metrics.json")
hist_raw = load_json(RUN_DIR / "metrics_history.json")
history = hist_raw["history"] if hist_raw and "history" in hist_raw else []

if not history and hist_raw and isinstance(hist_raw, list):
    history = hist_raw  # older format: bare list

if config is None:
    print("WARNING: config.json not found")
else:
    print("Config:")
    for k, v in config.items():
        print(f"  {k}: {v}")

if metrics is None:
    print("WARNING: metrics.json not found")
else:
    best = metrics.get("best_epoch", "?")
    apm1 = metrics.get("accuracy_pm1", float("nan"))
    exact = metrics.get("exact_accuracy", float("nan"))
    mae  = metrics.get("mean_absolute_error", float("nan"))
    print(f"\nBest epoch : {best}")
    print(f"Accuracy±1 : {apm1:.5f}")
    print(f"Exact acc  : {exact:.5f}")
    print(f"MAE        : {mae:.5f}")
    pcr = metrics.get("per_class_recall", {})
    print("Per-class recall:", {CLASS_NAMES[int(k)]: round(v, 4) for k, v in pcr.items()})

print(f"\nEpochs in history: {len(history)}")


## Training Curves

In [ ]:
if not history:
    print("No history available.")
else:
    epochs    = [h["epoch"] for h in history]
    loss      = [h.get("train_loss", float("nan")) for h in history]
    apm1_hist = [h.get("accuracy_pm1", float("nan")) for h in history]
    exact_hist= [h.get("exact_accuracy", float("nan")) for h in history]
    mae_hist  = [h.get("mean_absolute_error", float("nan")) for h in history]
    best_ep   = metrics.get("best_epoch") if metrics else None

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].plot(epochs, loss, marker="o", color="#4c72b0", linewidth=2)
    axes[0].set_title("Train loss per epoch")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, apm1_hist, marker="o", color="#1a9850", linewidth=2, label="Accuracy±1")
    axes[1].plot(epochs, exact_hist, marker="s", color="#4c72b0", linewidth=2, linestyle="--", label="Exact")
    if best_ep in epochs:
        axes[1].axvline(best_ep, color="red", linestyle=":", linewidth=1.5, label=f"best={best_ep}")
    axes[1].set_title("Validation accuracy per epoch")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Rate")
    axes[1].set_ylim(0, 1); axes[1].legend(frameon=False); axes[1].grid(alpha=0.3)

    axes[2].plot(epochs, mae_hist, marker="o", color="#d73027", linewidth=2)
    if best_ep in epochs:
        axes[2].axvline(best_ep, color="red", linestyle=":", linewidth=1.5)
    axes[2].set_title("Validation MAE per epoch")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("MAE")
    axes[2].grid(alpha=0.3)

    fig.suptitle(f"{RUN_ID}  —  training curves", fontsize=11)
    fig.tight_layout()
    plt.show()


## Per-Class Recall Trajectory

In [ ]:
if not history:
    print("No history available.")
else:
    recall_by_class = {i: [] for i in range(5)}
    for h in history:
        pcr = h.get("per_class_recall", {})
        for i in range(5):
            recall_by_class[i].append(pcr.get(str(i), float("nan")))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for i in range(5):
        axes[0].plot(epochs, recall_by_class[i], marker="o", linewidth=2,
                     color=CLASS_COLORS[i], label=CLASS_NAMES[i])
    if best_ep in epochs:
        axes[0].axvline(best_ep, color="red", linestyle=":", linewidth=1.5, label=f"best={best_ep}")
    axes[0].set_title("Per-class recall over training")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Recall")
    axes[0].set_ylim(-0.05, 1.05); axes[0].legend(frameon=False, fontsize=9)
    axes[0].grid(alpha=0.3)

    # Best-epoch bar chart
    if metrics:
        best_pcr = metrics.get("per_class_recall", {})
        recalls = [best_pcr.get(str(i), 0.0) for i in range(5)]
        bars = axes[1].bar(range(5), recalls, color=CLASS_COLORS, edgecolor="black", linewidth=0.5)
        axes[1].set_xticks(range(5), CLASS_NAMES, rotation=15, ha="right")
        axes[1].set_ylabel("Recall"); axes[1].set_ylim(0, 1)
        axes[1].set_title(f"Per-class recall at best epoch ({metrics.get('best_epoch')})")
        for bar, r in zip(bars, recalls):
            axes[1].text(bar.get_x() + bar.get_width()/2, r + 0.02, f"{r:.3f}",
                         ha="center", fontsize=9)
        axes[1].grid(axis="y", alpha=0.3)

    fig.suptitle(f"{RUN_ID}  —  per-class recall", fontsize=11)
    fig.tight_layout()
    plt.show()


## Confusion Matrix (Best Epoch)

In [ ]:
if metrics and "confusion_matrix" in metrics:
    cm = np.array(metrics["confusion_matrix"])
    n  = cm.shape[0]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    im = axes[0].imshow(cm, cmap="Blues")
    axes[0].set_title("Confusion matrix (counts)")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    axes[0].set_xticks(range(n), CLASS_NAMES[:n], rotation=20, ha="right")
    axes[0].set_yticks(range(n), CLASS_NAMES[:n])
    thresh = cm.max() * 0.4
    for i in range(n):
        for j in range(n):
            axes[0].text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                         color="white" if cm[i,j] > thresh else "black", fontsize=8)
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    im2 = axes[1].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    axes[1].set_title("Confusion matrix (row-normalised = recall per class)")
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
    axes[1].set_xticks(range(n), CLASS_NAMES[:n], rotation=20, ha="right")
    axes[1].set_yticks(range(n), CLASS_NAMES[:n])
    for i in range(n):
        for j in range(n):
            axes[1].text(j, i, f"{cm_norm[i,j]:.2f}", ha="center", va="center",
                         color="white" if cm_norm[i,j] > 0.5 else "black", fontsize=9)
    fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

    fig.suptitle(f"{RUN_ID}  —  confusion matrix", fontsize=11)
    fig.tight_layout()
    plt.show()
else:
    print("No confusion_matrix in metrics.json")


## Val Probs Summary (if available)

In [ ]:
if not VAL_PROBS.exists():
    print(f"val_probs not found: {VAL_PROBS}")
else:
    payload = np.load(VAL_PROBS, allow_pickle=True)
    y_true  = payload["y_true"].astype(np.int16)
    y_pred  = payload["y_pred"].astype(np.int16)
    probs   = payload["probs"].astype(np.float32)
    patch_ids = payload["patch_ids"]

    valid = (y_true >= 0) & (y_true <= 4) & (y_pred >= 0) & (y_pred <= 4)
    diff  = np.abs(y_true - y_pred)
    total = int(valid.sum())

    global_apm1  = float((diff <= 1)[valid].mean())
    global_exact = float((diff == 0)[valid].mean())
    global_mae   = float(diff[valid].mean())

    print(f"Patches        : {len(patch_ids)}")
    print(f"Valid pixels   : {total:,}")
    print(f"Accuracy ±1    : {global_apm1:.5f}")
    print(f"Exact accuracy : {global_exact:.5f}")
    print(f"MAE            : {global_mae:.5f}")
    print(f"Ignored pixels : {int((~((y_true>=0)&(y_true<=4))).sum()):,}")

    # Per-class truth vs pred distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    true_counts = np.bincount(y_true[(y_true>=0)&(y_true<=4)].ravel(), minlength=5)
    pred_counts = np.bincount(y_pred[(y_pred>=0)&(y_pred<=4)].ravel(), minlength=5)
    x = np.arange(5); w = 0.38
    axes[0].bar(x - w/2, true_counts, width=w, color=CLASS_COLORS, alpha=0.85, label="Truth")
    axes[0].bar(x + w/2, pred_counts, width=w, color="black", alpha=0.45, label="Pred")
    axes[0].set_xticks(x, CLASS_NAMES, rotation=15, ha="right")
    axes[0].set_title("Class distribution: truth vs prediction")
    axes[0].legend(frameon=False)
    axes[0].grid(axis="y", alpha=0.3)

    # Patch-level Acc±1 histogram
    valid_patch = valid.sum(axis=(1,2))
    patch_apm1  = np.where(valid_patch > 0,
                           ((diff<=1)&valid).sum(axis=(1,2)) / (valid_patch + 1e-9), np.nan)
    axes[1].hist(patch_apm1[~np.isnan(patch_apm1)], bins=25,
                 color="#1a9850", alpha=0.75, edgecolor="white", linewidth=0.3)
    axes[1].axvline(float(np.nanmedian(patch_apm1)), color="red", linestyle="--", linewidth=1.5,
                    label=f"median={np.nanmedian(patch_apm1):.3f}")
    axes[1].set_title("Patch-level Accuracy±1 distribution")
    axes[1].set_xlabel("Patch Accuracy±1"); axes[1].set_ylabel("Patch count")
    axes[1].legend(frameon=False)
    axes[1].grid(alpha=0.3)

    fig.suptitle(f"{RUN_ID}  —  val_probs summary", fontsize=11)
    fig.tight_layout()
    plt.show()


## All-Runs Comparison Table

In [ ]:
rows = []
for run_dir in sorted(RUNS_DIR.iterdir()):
    if not run_dir.is_dir():
        continue
    m = load_json(run_dir / "metrics.json")
    c = load_json(run_dir / "config.json")
    if m is None:
        continue
    pcr = m.get("per_class_recall", {})
    row = {
        "run_id"     : run_dir.name,
        "model"      : (c or {}).get("model", "?"),
        "loss"       : (c or {}).get("loss", "?"),
        "temporal"   : (c or {}).get("temporal_mode", "?"),
        "best_epoch" : m.get("best_epoch"),
        "acc_pm1"    : round(m.get("accuracy_pm1", float("nan")), 5),
        "exact"      : round(m.get("exact_accuracy", float("nan")), 5),
        "mae"        : round(m.get("mean_absolute_error", float("nan")), 4),
        "r0_vlow"    : round(pcr.get("0", float("nan")), 3),
        "r1_low"     : round(pcr.get("1", float("nan")), 3),
        "r2_med"     : round(pcr.get("2", float("nan")), 3),
        "r3_high"    : round(pcr.get("3", float("nan")), 3),
        "r4_vhigh"   : round(pcr.get("4", float("nan")), 3),
        "val_probs"  : (VAL_PREDS_DIR / f"{run_dir.name}_val_probs.npz").exists(),
    }
    rows.append(row)

if rows:
    df = pd.DataFrame(rows).sort_values("acc_pm1", ascending=False).reset_index(drop=True)
    pd.set_option("display.max_colwidth", 55)
    display(df)
else:
    print("No completed runs found under", RUNS_DIR)
